# Bayesian Optimization for Black-Box Functions (Submission 8)

This notebook implements the advanced **"Final Frontier"** Bayesian Optimization framework configured for **Round 8 Queries** across 8 independent black-box environments.

### Core Optimization Architecture & Strategy for Round 8:
* **Matern 5/2 Smoothness-Flexibility Balance:** Employs a robust Matern 5/2 kernel combined with a high-rigor fitting configuration (**50 restarts inside the GPR engine**) to perfectly tune localized Automatic Relevance Determination (ARD) length-scales.
* **Maximum Global Acquisition Surface Search Intensity:** To strictly mitigate getting caught in local non-convex EI traps, the Expected Improvement (EI) surface is exhaustively optimized via **150 randomized multi-starts** using the L-BFGS-B local solver.
* **"Final Frontier" Exploration Strategy Matrix ($\xi$-Mapping):**
  * **$\xi = 0.5000$ (Maximum Exploration Shock):** Configured for Channels 1, 3, 4, and 6 to forcefully escape stagnant flat zones or deep negative landscape traps.
  * **$\xi = 0.0500$ (Noisy-Landscape Tracking):** Maintained for Channel 2 alongside data regularizer $\alpha = 10^{-2}$ to distinguish physical process signal from stochastic observation variance.
  * **$\xi = 0.00001$ (Precision Micro-Exploitation Basins):** Applied to Channels 5 and 7 to execute local microscopic adjustments and reclaim historical structural highs.
  * **$\xi = 0.0000$ (Absolute Pure Exploitation Peak Tracking):** Dedicated strictly to Channel 8 to leverage the maximum multi-start search intensity and push past the current 9.914 peak directly onto the 10.0 target milestone boundary.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enable inline plotting for downstream validation charts
%matplotlib inline 

print("Environment initialized. Round 8 Final Frontier engine standing by.")

## Definition of Object-Oriented Bayesian Optimizer

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Configures regularizer parameters for noise mitigation.
        """
        # alpha=1e-2 for noisy F2, 1e-6 for others to maintain high fidelity
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads cumulative multi-round historical parameters and fits the precise GPR model.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        # Using Matern 2.5 for its balance of smoothness and flexibility
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        # High rigor to ensure length scales perfectly capture dimension importance
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha, 
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surface for a targeted exploration jitter (xi).
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Maximizes Expected Improvement across an intensive array of 150 local search multi-starts.
        """
        best_x = None
        max_ei = -1
        # Increased to 150 restarts to prevent getting stuck in local EI minima
        for _ in range(150):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Sequential Execution Optimization Pipeline

Loops through active function channels, tracks and references unique directory sets, models the functional space, and writes query parameter outputs to the standardized submission template format.

In [ ]:
output_file = "submission_8_results.txt"

# xi_map Logic: "Final Frontier"
# F1, F3, F4, F6: Max exploration (0.5) to escape stagnant/negative regions.
# F5, F7: Precision exploitation (0.00001) to recover historical highs.
# F8: Pure exploitation (0.0) with high-rigor search to hit the 10.0 target.
xi_map = {
    1: 0.5000, 
    2: 0.0500, 
    3: 0.5000, 
    4: 0.5000, 
    5: 0.00001, 
    6: 0.5000, 
    7: 0.00001, 
    8: 0.0000  
}

print("Generating Submission 8: Final Frontier Calibration...")
print("-" * 55)

notebook_results = {}

for i in range(1, 8 + 1):
    func_dir = f"function_{i}"
    
    if not os.path.exists(func_dir):
        print(f"Warning: Execution target '{func_dir}' missing from notebook working path.")
        continue
        
    optimizer = BayesianOptimizer(is_noisy=(i == 2))
    optimizer.load_and_fit(func_dir)
    
    current_xi = xi_map[i]
    next_point = optimizer.propose_next_point(xi=current_xi)
    formatted_point = "-".join([f"{val:.6f}" for val in next_point])
    
    notebook_results[f"Function {i}"] = (formatted_point, current_xi)
    print(f"Function {i}: {formatted_point} (xi={current_xi})")

# Export collected parameters out to output document context
with open(output_file, "w") as f:
    for i in range(1, 9):
        key = f"Function {i}"
        if key in notebook_results:
            f.write(f"{key}: {notebook_results[key][0]}\n")

print("-" * 55)
print(f"Submission 8 successfully exported onto filesystem path: {output_file}")

### Final Proposed Coordinates Summary Matrix

In [ ]:
print(f"| {'Target Channel':15} | {'Exploration (xi)':18} | {'Proposed Query Coordinates (Round 8)':55} |")
print(f"| {'-'*15} | {'-'*18} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<18} | {coords:55} |")